# Batch Cellpose-SAM Segmentation for Weekly 3D TIFF Images

This notebook:

- Reads only TIFF files directly inside the target folder.
- Ignores TIFF files in subfolders such as `crop_128`, `crop_256`, and `crop_512`.
- Processes only files whose names end with `_R.tif` or `_R.tiff`.
- Runs Cellpose-SAM in 3D mode.
- Saves masks into `cellposeSAM_masks` with the suffix `_cp_masks.tif`.


In [1]:
from pathlib import Path

from tqdm.auto import tqdm
from cellpose import io as cp_io
from cellpose import models
from cellpose.io import imread_3D
import wslPath
import numpy as np



/mnt/d/Codex_folder/molecular_tracking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

The Windows folder

`D:\\_data\\_newAAV_2026\\weekly_registration_test`

is accessed in WSL as

`/mnt/d/_data/_newAAV_2026/weekly_registration_test`.


In [2]:
input_dir = r"D:\_data\_newAAV_2026\weekly_registration_test"
input_dir = Path(wslPath.to_posix(input_dir))

output_dir = input_dir / "cellposeSAM_masks"
output_dir.mkdir(parents=True, exist_ok=True)

# Set to True to rerun files whose mask outputs already exist.
overwrite_existing = False

# Cellpose parameters
min_size = 100


## Find top-level red-channel TIFF files

`Path.iterdir()` checks only the immediate contents of the folder, so subfolders are not searched.


In [4]:
image_paths = sorted(
    path
    for path in input_dir.iterdir()
    if (
        path.is_file()
        and path.suffix.lower() in {".tif", ".tiff"}
        and path.stem.lower().endswith("_r")
    )
)

if not image_paths:
    raise FileNotFoundError(
        f"No top-level TIFF files ending in '_R' were found in:\n{input_dir}"
    )

print(f"Found {len(image_paths)} red-channel image(s):")
for path in image_paths:
    print(f"  {path.name}")


Found 24 red-channel image(s):
  20260511_R.tif
  20260512_R.tif
  20260513_R.tif
  20260514_R.tif
  20260515_R.tif
  20260518_R.tif
  20260519_R.tif
  20260521_R.tif
  20260522_R.tif
  20260526_R.tif
  20260527_R.tif
  20260528_R.tif
  20260529_R.tif
  20260601_R.tif
  20260602_R.tif
  20260603_R.tif
  20260604_R.tif
  20260605_R.tif
  20260608_R.tif
  20260617_R.tif
  20260625_R.tif
  20260701_R.tif
  20260708_R.tif
  20260709_R.tif


## Initialize Cellpose-SAM


In [5]:
model = models.CellposeModel(
    gpu=True,
    pretrained_model="cpsam_v2",
)
# cp_io.logger_setup()

## Run 3D segmentation

For an input file such as `20260511_R.tif`, the saved output will be:

`cellposeSAM_masks/20260511_R_cp_masks.tif`


In [ ]:
failed_files = []
processed_files = []
skipped_files = []

for image_path in tqdm(image_paths, desc="Cellpose-SAM segmentation"):
    expected_output = output_dir / f"{image_path.stem}_cp_masks.tif"

    if expected_output.exists() and not overwrite_existing:
        skipped_files.append(image_path.name)
        tqdm.write(f"Skipping existing: {expected_output.name}")
        continue

    try:
        tqdm.write(f"Processing: {image_path.name}")

        loaded_image = imread_3D(image_path.as_posix())

        masks, flows, styles = model.eval(
            loaded_image,
            do_3D=True,
            z_axis=0,
            channel_axis=3,
            min_size=min_size,
        )
        print(f"  Found {np.max(masks)} mask(s) in {image_path.name}")
        cp_io.save_masks(
            images=loaded_image,
            masks=masks,
            flows=flows,
            file_names=image_path.as_posix(),
            png=False,
            tif=True,
            suffix="_cp_masks",
            savedir=output_dir.as_posix(),
        )

        processed_files.append(image_path.name)
        tqdm.write(f"Saved: {expected_output.name}")

    except Exception as error:
        failed_files.append((image_path.name, str(error)))
        tqdm.write(f"FAILED: {image_path.name}")
        tqdm.write(f"  {error}")


Cellpose-SAM segmentation:   0%|          | 0/24 [00:00<?, ?it/s]

Skipping existing: 20260511_R_cp_masks.tif
Processing: 20260512_R.tif


Cellpose-SAM segmentation:   8%|▊         | 2/24 [05:04<55:54, 152.50s/it]

  Found 7078 mask(s) in 20260512_R.tif
Saved: 20260512_R_cp_masks.tif
Processing: 20260513_R.tif


Cellpose-SAM segmentation:  12%|█▎        | 3/24 [10:10<1:15:44, 216.41s/it]

...

Cellpose-SAM segmentation:  46%|████▌     | 11/24 [51:21<1:06:24, 306.50s/it]

  Found 7395 mask(s) in 20260527_R.tif
Saved: 20260527_R_cp_masks.tif
Processing: 20260528_R.tif


Cellpose-SAM segmentation:  50%|█████     | 12/24 [56:33<1:01:37, 308.10s/it]

  Found 7533 mask(s) in 20260528_R.tif
Saved: 20260528_R_cp_masks.tif
Processing: 20260529_R.tif


100%|██████████| 41/41 [00:00<00:00, 92.90it/s]


## Summary


In [ ]:
print("Finished.")
print(f"Output folder: {output_dir}")
print(f"Processed: {len(processed_files)}")
print(f"Skipped:   {len(skipped_files)}")
print(f"Failed:    {len(failed_files)}")

if failed_files:
    print("\nFailed files:")
    for filename, error in failed_files:
        print(f"  {filename}: {error}")
